# Optimized PyTorch CNN with Optuna

In this notebook, we optimize a convolutional neural network (CNN) for the CIFAR-10 dataset using Optuna for hyperparameter tuning.

## Notebook set-up

### Imports

In [ ]:
# Standard library imports
import pickle

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.optim as optim
from torchvision import datasets, transforms

# Package imports
import image_classification_tools.pytorch.data as data_utils
import image_classification_tools.pytorch.evaluation as eval_utils
import image_classification_tools.pytorch.hyperparameter_optimization as optimization
import image_classification_tools.pytorch.plotting as plots
import image_classification_tools.pytorch.training as training

# Local imports
import configuration as config

### Fixed hyperparameters

In [ ]:
# Optuna settings
run_optimization = True  # Run optimization (True) or load results for evaluation (False)
start_new_study = False   # Clear results/restart (True) or resume previous run (False) 
validation_size = 10000
n_trials = 200           # Number of optimization trials
n_epochs_per_trial = 200 # Epochs per trial
n_epochs_final = 200     # Epochs for final model training with optimized hyperparameters
print_every = 10         # Print training progress every n epochs

## 1. Visualize CIFAR-10 sample images

CIFAR-10 contains 32x32 color images (3 channels) across 10 classes.

In [ ]:
# Define transform (RGB)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# Get a sample dataset for visualization
sample_dataset = datasets.CIFAR10(
    root=config.DATA_DIR,
    train=True,
    transform=transform
)

# Plot first 10 images from the training dataset
fig, axes = plots.plot_sample_images(sample_dataset, config.CLASS_NAMES)
plt.show()

## 2. Optuna hyperparameter optimization

We use `create_objective()` to generate an objective function that Optuna will optimize.
The function samples hyperparameters, creates and trains a model, and returns validation accuracy.

### 2.1. Define hyperparameter search space

In [ ]:
# Define hyperparameter search space
search_space = {
    'batch_size': [64, 128, 256, 512],
    'n_conv_blocks': (1, 8),               # Number of conv blocks
    'initial_filters': [16, 32, 64, 128],  # Filters in first block (doubles each block)
    'n_fc_layers': (1, 5),                 # Number of FC layers in classifier
    'conv_dropout_rate': (0.1, 0.5),       # Conv block dropout
    'fc_dropout_rate': (0.3, 0.7),         # FC layer dropout
    'learning_rate': (1e-5, 1e-2, 'log'),
}

print('Hyperparameter search space:')

for key, value in search_space.items():
    print(f'  {key}: {value}')

### 2.2. Create objective function

In [ ]:
# Create objective function for Optuna
objective = optimization.create_objective(
    data_dir=config.DATA_DIR,
    transform=transform,
    n_epochs=n_epochs_per_trial,
    device=config.DEVICE,
    num_classes=len(config.CLASS_NAMES),
    search_space=search_space,
    early_stopping_patience=5
)

### 2.3. Run optimization

In [ ]:
%%time

if run_optimization:
    print('Running hyperparameter optimization...')

    # Delete existing study if desired & it exists
    if start_new_study == True:

        print('Starting new study')
        try:
            optuna.delete_study(study_name='cnn_optimization', storage=config.OPTUNA_STORAGE_URL)
            print('Deleted existing study')

        except KeyError:
            print('No existing study found')

    else:
        if config.OPTUNA_DB_PATH.exists():
            print(f'Resuming study {config.OPTUNA_DB_PATH}')

        else:
            print(f'No prior results found at {config.OPTUNA_DB_PATH}, starting new study')

    # Create Optuna study with SQLite storage (maximize validation accuracy)
    study = optuna.create_study(
        direction='maximize',
        study_name='cnn_optimization',
        storage=config.OPTUNA_STORAGE_URL,
        load_if_exists=True,  # Resume if study already exists
        pruner=None #optuna.pruners.MedianPruner(n_warmup_steps=5)
    )

    print(f'Study stored at: {config.OPTUNA_DB_PATH}')

    # Run optimization
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

else:

    # Load results from disk
    study = optuna.load_study(
        study_name='cnn_optimization',
        storage=config.OPTUNA_STORAGE_URL
    )

    print(f'Study loaded from: {config.OPTUNA_DB_PATH}')

print(f'\nBest validation accuracy: {study.best_trial.value:.2f}%')

for key, value in study.best_trial.params.items():
    print(f' - {key}: {value}')

print()


### 2.4. Visualize optimization results

In [ ]:
fig, axes = plots.plot_optimization_results(study)
plt.show()

## 3. Train final model with best hyperparameters

In [ ]:
# Extract best hyperparameters
best_params = study.best_trial.params

print('Best hyperparameters:')
for key, value in best_params.items():
    print(f'  {key}: {value}')

### 3.1. Re-create dataloader with winning batch size

In [ ]:
# Recreate data loaders with best batch size
best_batch_size = best_params['batch_size']

# Load datasets
train_dataset = data_utils.load_dataset(
    data_source=datasets.CIFAR10,
    transform=transform,
    root=config.DATA_DIR,
    train=True
)

test_dataset = data_utils.load_dataset(
    data_source=datasets.CIFAR10,
    transform=transform,
    root=config.DATA_DIR,
    train=False
)

# Prepare splits
train_dataset, val_dataset, test_dataset = data_utils.prepare_splits(
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    val_size=validation_size
)

# Create dataloaders
train_loader, val_loader, test_loader = data_utils.create_dataloaders(
    train_dataset, val_dataset, test_dataset,
    batch_size=best_batch_size,
    preload_to_memory=True,
    device=config.DEVICE
)

### 3.2. Create optimized model

In [ ]:
# Create model with best hyperparameters
best_model = optimization.create_cnn(
    n_conv_blocks=best_params['n_conv_blocks'],
    initial_filters=best_params['initial_filters'],
    n_fc_layers=best_params['n_fc_layers'],
    conv_dropout_rate=best_params['conv_dropout_rate'],
    fc_dropout_rate=best_params['fc_dropout_rate'],
    num_classes=len(config.CLASS_NAMES),
    in_channels=3
).to(config.DEVICE)

# Create optimizer with best learning rate (fixed Adam optimizer)
best_optimizer = optim.Adam(
    best_model.parameters(), 
    lr=best_params['learning_rate']
)

# Set cross-entropy loss
criterion = torch.nn.CrossEntropyLoss()

# Get total trainable parameters
trainable_params = sum(p.numel() for p in best_model.parameters() if p.requires_grad)

print(f'\n{best_model}')
print(f'\nTotal parameters: {trainable_params:,}')

### 3.2. Train optimized model

In [ ]:
%%time

history = training.train_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=best_optimizer,
    device=config.DEVICE,
    epochs=n_epochs_final,
    print_every=print_every,
    enable_early_stopping=True,
    early_stopping_patience=10
)

print()

### 3.3. Learning curves

In [ ]:
fig, axes = plots.plot_learning_curves(history)
plt.show()

## 4. Evaluate optimized model on test set

### 4.1. Calculate test accuracy

In [ ]:
test_accuracy, predictions, true_labels = eval_utils.evaluate_model(
    best_model,
    test_loader
)

print(f'Test accuracy: {test_accuracy:.2f}%')

### 4.2. Per-class accuracy

In [ ]:
# Calculate per-class accuracy
class_correct = {name: 0 for name in config.CLASS_NAMES}
class_total = {name: 0 for name in config.CLASS_NAMES}

for pred, true in zip(predictions, true_labels):

    class_name = config.CLASS_NAMES[true]
    class_total[class_name] += 1

    if pred == true:
        class_correct[class_name] += 1

print('Per-class accuracy:')
print('-' * 30)

for name in config.CLASS_NAMES:
    acc = 100 * class_correct[name] / class_total[name]
    print(f'{name:12s}: {acc:.2f}%')

### 4.3. Confusion matrix

In [ ]:
fig, ax = plots.plot_confusion_matrix(true_labels, predictions, config.CLASS_NAMES)
plt.show()

### 4.4. Predicted class probability distributions

In [ ]:
# Get predicted probabilities for all test samples
best_model.eval()
all_probs = []

with torch.no_grad():
    for images, _ in test_loader:
        outputs = best_model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)

# Plot probability distributions
fig, axes = plots.plot_class_probability_distributions(all_probs, config.CLASS_NAMES)
plt.show()

### 4.5. Evaluation curves

In [ ]:
fig, (ax1, ax2) = plots.plot_evaluation_curves(true_labels, all_probs, config.CLASS_NAMES)
plt.show()

## 5. Save optimized model and hyperparameters

In [ ]:
# Save trained model
model_path = config.MODELS_DIR / 'optimized_cnn.pth'
torch.save(best_model, model_path)

print(f'Model saved to: {model_path}')
print(f'Test accuracy: {test_accuracy:.2f}%')

## 6. Save test results for comparison

In [ ]:
# Save performance results
results_path = config.RESULTS_DIR / 'optimized_cnn_results.pkl'

# Count model parameters
total_params = sum(p.numel() for p in best_model.parameters())
trainable_params = sum(p.numel() for p in best_model.parameters() if p.requires_grad)

# Create results dictionary
results_dict = {
    'true_labels': true_labels,
    'predictions': predictions,
    'all_probs': all_probs,
    'test_accuracy': test_accuracy,
    'total_params': total_params,
    'trainable_params': trainable_params
}

# Save results
with open(results_path, 'wb') as f:
    pickle.dump(results_dict, f)

print(f'Test results saved to: {results_path}')
print(f'  - Test accuracy: {test_accuracy:.2f}%')
print(f'  - Total parameters: {total_params:,}')
print(f'  - Trainable parameters: {trainable_params:,}')
